___
# Object Collaborations in Python
Gary Mitchell
March 18, 2020

---

Most object oriented implementations provide their functionality not through
the capabilities of a single class, but through collaborations among numerous
classes (specifically, between instances of classes). Collaboration is accomplished by *passing messages* and/or *making requests*.
This sounds much more elaborate than it is. The basic OOP mechanism for passing a 
message or making a request is the method call.

## A simple collaboration example

The following class diagram shows a very simple collaboration example:

<img src=Person_class.jpg>

The code below implements the Person class described
in the class diagram above:

In [7]:
class Person:
    def __init__(self, name):
        self.name = name
        self.friends = dict()
        
    def acceptGreeting(self, person):
        # Returning a msg - this is our ANSWER
        return (f"Hi {person.name}, it's {self.name}. I am here.")
        
    def addFriend(self, friend):
        if friend.name in self.friends:
            # a friend with that name already exists - cannot add
            return False
        else:
            # OK to add friend with that name
            self.friends[friend.name] = friend
            return True
        
    def greetFriends(self):
        # reach out to ALL friends
        if len(self.friends) == 0:
            print(f'My name is {self.name}. I have no friends.')
            return
            
        print(f'My name is {self.name} and I have {len(self.friends)} friends:')
        
        for k, v in self.friends.items():
            print(f'\tHi {k}. Are you there?')
            # Actual reaching out to the friend - i.e. here is where we send the msg
            print(f'\t{v.acceptGreeting(self)}')

In this example, we have a single class, Person. We'll proceed by creating three person
instances (i.e. *people*) that will communicate/collaborate with each other.

In [8]:
bob = Person('Bob')
carol = Person('Carol')
ted = Person('Ted')

Now that we have three "people", we'll create some friendships:

-   Bob considers both Carol and Ted as friends
-   Carol considers only Bob as a friend
-   Ted feels like he has no friends

We'll associate each Person with his/her friends using the addFriend method. Add friend
allows each Person object to *track* its friends. A Person can communicate with its *friends*
using the friend's object reference.

In [9]:
# establish Bob's friends
bob.addFriend(carol)
bob.addFriend(ted)

# establish Carol's friends
carol.addFriend(bob)

True

Now, each of our *people* will great his/her friends. Note that after a Person reaches out
to his/her friends, the friends reply. In this example, therefore, we have a message/reply
sequence that takes place.

In [10]:
bob.greetFriends()

carol.greetFriends()

ted.greetFriends()

My name is Bob and I have 2 friends:
	Hi Carol. Are you there?
	Hi Bob, it's Carol. I am here.
	Hi Ted. Are you there?
	Hi Bob, it's Ted. I am here.
My name is Carol and I have 1 friends:
	Hi Bob. Are you there?
	Hi Carol, it's Bob. I am here.
My name is Ted. I have no friends.


Communication between objects need not be two-way. 
It may be that only one object needs to know about the other. For example, consider
the relationship between a *jar* and the *pickles* it contains. 

![Jar and Pickles Class Diagram](Pickle_jar.jpg)

The jar must be aware of
the pickles, because it contains them. But the pickle is happy just existing. It doesn't
care (or need to know) whether it is in the jar or lying on a plate.

The code below implements Pickle and Jar as described in the class diagram.

In [11]:
class Pickle:
    # Pickles don't really have to know ANYTHING
    def __str__(self):
        return f'I am a happy pickle at {id(self)};'
    
class Jar:
    def __init__(self):
        self._contents = list()
        
    def add(self, obj):
        # the Jar doesn't really even have to know what type of Object it contains
        # as long as it knows that it does contain an object. Note that the 
        # Pickle instances will be "contained" by the Jar, just as real pickles are
        # physicall contained in jars.

        print(f'Adding pickle {id(obj)}')
        self._contents.append(obj)
        
    def empty(self):
        # empty the contents for the Jar, LIFO order
        while len(self._contents) > 0:
            obj = self._contents.pop()
            print(obj)
        
    def remove(self):
        if len(self._contents) > 0:
            # return the LAST object added to the jar
            return self._contents.pop()
        else:
            return None
        

Now we'll create a jar and place 10 pickles in the jar.

In [12]:
jar = Jar()

# add 10 pickles into the Jar
for i in range(10):
    jar.add(Pickle())
    

Adding pickle 1923832739360
Adding pickle 1923832740320
Adding pickle 1923832727648
Adding pickle 1923832732928
Adding pickle 1923832726544
Adding pickle 1923832726928
Adding pickle 1923832728080
Adding pickle 1923832740416
Adding pickle 1923832737296
Adding pickle 1923832739840


Let's remove a single pickle from the jar. In real life, jars are essentially
LIFO (last in first out) containers (i.e. the pickle on the top is easy to remove,
the one at the bottom is difficult or impossible to remove first).

In [13]:
print(jar.remove())

I am a happy pickle at 1923832739840;


Now, we'll empty the jar, removing the remaining pickles.

In [14]:
jar.empty()
        

I am a happy pickle at 1923832737296;
I am a happy pickle at 1923832740416;
I am a happy pickle at 1923832728080;
I am a happy pickle at 1923832726928;
I am a happy pickle at 1923832726544;
I am a happy pickle at 1923832732928;
I am a happy pickle at 1923832727648;
I am a happy pickle at 1923832740320;
I am a happy pickle at 1923832739360;


Observe that the *pickles* were completely unaware of the *jar* at all
during this interaction. How can we tell? The pickles *never* had a
*reference* to the jar:

-   Pickle objects do not have an attribute (i.e. instance variable) that
points to a jar
-   No pickle method accepts a Jar as an argument.

Thus, pickles are completely unware of jars at all times and the interactions
between pickles and jars are all one way (i.e. jars know about pickles, pickles
are unaware of jars).

## Takeaways
In order for objects to communicate/collaborate with other objects, they
must have *references* to those objects. The source of the reference indicates
whether a reference is permanent or temporary.

### Permanent relationships
If an object has a *permanent* relationship with another object, then it
will retain a reference to that object as an attribute. We saw this in the
examples above:

-   A Person object maintained references to its *friends*
-   A Jar object maintained references to the Pickle objects it contained.

In a permanent relationship, an object can collaborate with the related 
object at any time, simply by invoking a method on the object's reference.
in Person.greetFriends, we see the Person instance *talking* to each
of its friends by invoking the friend object's acceptGreeting method.

With whole-part relations (i.e. aggregations), we generally expect the 
*whole* to have references to its *parts*, but not necessarily the other
way around. 

However, two way relationships are certainly possible. In this
case, both objects must have a reference to the other. We see this in the
Person example. Bob has a reference to Carol as one of his friends. Carol
also has a reference to Bob as one of her friends. Therefore, both Carol
and Bob can initiate communication with the other person.

### Temporary relationships
If two objects have a temporary relation, they usually have awareness of
the relationship only for the duration of a method call. Thus the relationship
is transitory. 

In the Person example, Bob considers Ted a friend. But Ted feels he has no
friends (sad, I know). When Bob greets Ted, he is aware of Ted as a friend
(Ted exists in Bob's collection of friends). Bob can contact Ted at any time.

However, Ted is unaware of Bob
until Bob initiates the contact (by calleing acceptGreeting). Under normal
circumstances, Ted cannot contact Bob, because he has no reference to Bob.
When Bob contacts Ted, he makes himself known to Ted (Bob passes a 
reference to himself when he calls Ted's acceptGreeting method). For the 
duration of that contact/interaction (i.e. the duration of the method 
execution), Ted is aware of Bob and is able to communicate with Bob freely (
i.e. Ted is allowed to call any of Bob's public methods). Ted can also *answer*
Bob's query (i.e. he can return a value from his acceptGreeting method). 

Thus, in this example, Ted has a temporary relationship 
with Bob for the duration of the method call (and Bob has a permanent)
relationship with Ted.

### Whole-part relationships
Whole-part (i.e. assembly-part, container-contents, collection-
members) relationships are generally implemented by the whole
using a *container* class. Classes such as list, dictionary,
set, even tuple, are container classes because they can be
used to *contain* references to multiple other objects. In 
one-way interactions, this is typically all that's required.
If a two way interaction is required, then the parts will
need references to the wholes:

-   If the cardinality on the whole end of the relationship
is 1 or 0..1, then the part will have a single attribute 
for containing a reference to the whole
-   If the cardinality on the whole end is 0..*, then the
part will *also* have a container attribute that will be
used to contain references to multiple wholes.